In [7]:
!pip install transformers seqeval evaluate datasets


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\helen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Data Loading

In [8]:
from datasets import load_dataset

dataset_raw = load_dataset("lfcc/portuguese_ner")
dataset_raw

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [9]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

### Data Pre-Processing

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

In [11]:
inputs = tokenizer("As aulas de PLNEB são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])  #o tokenizer parte as palavras que nao conhece
print(tokens)

# o SEP serve para dizer que a frase acabou e a seguir vem outra

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', '##EB', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [13]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01'], ...])

In [16]:
tokens = ["as", "aulas", "plneb", "são", "interessantes", "!"]
inputs = tokenizer(tokens, is_split_into_words=True)

new_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"]) 
print(new_tokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [17]:
len(tokens), len(new_tokens)

(6, 10)

In [19]:
inputs.word_ids()

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [22]:
def align_labels_with_tokens(word_ids, labels):
    new_labels = []
    previous_word = None
    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100)  # diz para ignorar completamente esta label
        elif previous_word != word_id:
            new_labels.append(labels[word_id])
        else:
            new_labels.append(-100)
        previous_word = word_id
    return new_labels


def tokenize_dataset(dataset):
    res = []
    for row in dataset:
        inputs = tokenizer(row["tokens"], is_split_into_words=True)
        new_labels = align_labels_with_tokens(inputs.word_ids(), row["ner_tags"])
        inputs["labels"] = new_labels
        res.append(inputs)
    return res

train_dataset = tokenize_dataset(dataset_raw["train"])
test_dataset = tokenize_dataset(dataset_raw["test"])
len(train_dataset), len(test_dataset)


(3716, 930)

In [24]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


### Model Training

In [ ]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu

In [28]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained("neuralmind/bert-base-portuguese-cased")

ImportError: 
AutoModelForTokenClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
